# 小红书爆款文案生成 Agent — 企业级真实工具集成

本 Notebook 是 `rednote.ipynb` 教学版本的企业级实现，将三个模拟工具替换为真实实现：

1. `mock_search_web` → `real_search_web`：调用 Tavily 搜索 API
2. `mock_query_product_database` → `real_query_product_database`：Milvus 向量数据库语义检索
3. `mock_generate_emoji` → `real_generate_emoji`：DeepSeek LLM 智能生成表情符号

## 1. 配置与环境管理

In [ ]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()

# ========== 日志配置 ==========
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    filename="./rednote-agent.log",
    filemode="a",
    encoding="utf-8",
    force=True,
)
logger = logging.getLogger("rednote-agent")

# ========== 全局常量 ==========
MILVUS_URI = "http://localhost:19530"
PRODUCT_COLLECTION = "product_collection"
SIMILARITY_THRESHOLD = 0.5
DEFAULT_EMOJIS = ["✨", "🔥", "💖", "💯", "🎉"]
DEEPSEEK_MODEL = "deepseek-chat"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-zh-v1.5"

# ========== 环境变量验证 ==========
REQUIRED_ENV_VARS = {
    "DEEPSEEK_API_KEY": "DeepSeek API 密钥",
    "TAVILY_API_KEY": "Tavily 搜索 API 密钥",
}

def validate_env_vars():
    """验证所有必需的环境变量是否已设置。"""
    missing = []
    for var, desc in REQUIRED_ENV_VARS.items():
        if not os.getenv(var):
            missing.append(f"  - {var}: {desc}")
    if missing:
        raise ValueError(
            "缺少以下必需的环境变量:\n" + "\n".join(missing)
        )

validate_env_vars()
logger.info("环境变量验证通过")

## 2. 客户端初始化

In [ ]:
from openai import OpenAI
from pymilvus import model

# ========== DeepSeek LLM 客户端 ==========
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
)
logger.info("DeepSeek LLM 客户端初始化完成")

# ========== Embedding 模型 ==========
embedding_model = model.dense.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME,
    device='cpu'
)
logger.info("Embedding 模型初始化完成")

## 3. 商品数据初始化流水线

In [ ]:
from pymilvus import MilvusClient

# 新增商品的相似度阈值：低于此值认为是新商品，需要写入
UPSERT_SIMILARITY_THRESHOLD = 0.85

def init_product_database(products: list[dict]):
    """将商品数据增量写入 Milvus 向量数据库。

    逻辑：
      - 如果 Collection 不存在，创建并全量写入。
      - 如果 Collection 已存在，逐条检查：用商品文本向量在库中搜索，
        相似度 >= UPSERT_SIMILARITY_THRESHOLD 视为已存在（跳过），
        相似度 < UPSERT_SIMILARITY_THRESHOLD 视为新商品（新增）。

    Args:
        products: 商品列表，每条包含 name, ingredients, effects, target_audience, specs 字段
    """
    # 验证 Schema
    required_fields = ["name", "ingredients", "effects", "target_audience", "specs"]
    for i, p in enumerate(products):
        for field in required_fields:
            if not p.get(field):
                raise ValueError(f"商品 #{i} 缺少必需字段 '{field}' 或字段为空")

    logger.info(f"[Product DB] 开始初始化，共 {len(products)} 条商品数据")
    milvus_client = MilvusClient(uri=MILVUS_URI)

    # 将商品信息拼接为完整文本用于 embedding
    texts = [
        f"{p['name']}：核心成分为{p['ingredients']}，功效为{p['effects']}，"
        f"适用人群为{p['target_audience']}，规格为{p['specs']}。"
        for p in products
    ]
    embeddings = embedding_model.encode_documents(texts)
    embedding_dim = len(embeddings[0])

    collection_exists = milvus_client.has_collection(PRODUCT_COLLECTION)

    # 如果已有 Collection 的 metric_type 不是 COSINE，删除重建
    if collection_exists:
        col_info = milvus_client.describe_collection(collection_name=PRODUCT_COLLECTION)
        # pymilvus 返回的 metric_type 可能在 index 描述中，也可能直接在 collection 描述中
        # 通过尝试搜索来检测，或者直接检查 index params
        indexes = milvus_client.list_indexes(collection_name=PRODUCT_COLLECTION)
        need_rebuild = False
        if indexes:
            idx_info = milvus_client.describe_index(collection_name=PRODUCT_COLLECTION, index_name=indexes[0])
            current_metric = idx_info.get('metric_type', '')
            if current_metric and current_metric != 'COSINE':
                need_rebuild = True
                logger.warning(f"[Product DB] 已有 Collection metric_type={current_metric}，需要重建为 COSINE")
        if need_rebuild:
            milvus_client.drop_collection(PRODUCT_COLLECTION)
            collection_exists = False
            logger.info("[Product DB] 已删除旧 Collection，将重新创建")

    if not collection_exists:
        # Collection 不存在 → 创建并全量写入
        milvus_client.create_collection(
            collection_name=PRODUCT_COLLECTION,
            dimension=embedding_dim,
            metric_type="COSINE",
            consistency_level="Strong",
        )
        data = [
            {"id": i, "vector": embeddings[i], "text": texts[i]}
            for i in range(len(products))
        ]
        milvus_client.insert(collection_name=PRODUCT_COLLECTION, data=data)
        milvus_client.flush(collection_name=PRODUCT_COLLECTION)
        logger.info(f"[Product DB] 新建 Collection，写入 {len(products)} 条商品数据")
        return

    # Collection 已存在 → 增量写入：逐条检查相似度
    # 先获取当前最大 id，用于分配新 id
    existing_count = milvus_client.query(
        collection_name=PRODUCT_COLLECTION,
        filter="",
        output_fields=["id"],
        limit=10000,
    )
    next_id = max((r["id"] for r in existing_count), default=-1) + 1

    new_data = []
    skipped = 0
    for idx, (text, emb) in enumerate(zip(texts, embeddings)):
        search_res = milvus_client.search(
            collection_name=PRODUCT_COLLECTION,
            data=[emb],
            limit=1,
            search_params={"metric_type": "COSINE", "params": {}},
            output_fields=["text"],
        )
        if search_res and search_res[0]:
            top_score = search_res[0][0]["distance"]
            if top_score >= UPSERT_SIMILARITY_THRESHOLD:
                logger.info(
                    f"[Product DB] 跳过已存在商品 '{products[idx]['name']}' "
                    f"(相似度 {top_score:.3f} >= {UPSERT_SIMILARITY_THRESHOLD})"
                )
                skipped += 1
                continue
        new_data.append({"id": next_id, "vector": emb, "text": text})
        next_id += 1

    if new_data:
        milvus_client.insert(collection_name=PRODUCT_COLLECTION, data=new_data)
        milvus_client.flush(collection_name=PRODUCT_COLLECTION)
        logger.info(f"[Product DB] 增量写入 {len(new_data)} 条新商品，跳过 {skipped} 条已存在商品")
    else:
        logger.info(f"[Product DB] 所有 {skipped} 条商品均已存在，无需写入")

In [ ]:
# ========== 示例商品数据 ==========
products = [
    {
        "name": "深海蓝藻保湿面膜",
        "ingredients": "深海蓝藻提取物，富含多糖和氨基酸",
        "effects": "深层补水、修护肌肤屏障、舒缓敏感泛红",
        "target_audience": "所有肤质，尤其适合干燥、敏感肌",
        "specs": "25ml*5片"
    },
    {
        "name": "美白精华",
        "ingredients": "烟酰胺和VC衍生物",
        "effects": "提亮肤色、淡化痘印、改善暗沉",
        "target_audience": "需要均匀肤色的人群",
        "specs": "30ml"
    }
]

# 初始化商品数据到 Milvus
init_product_database(products)

## 4. 真实工具函数实现

In [ ]:
from tavily import TavilyClient

# ========== 搜索结果向量库配置 ==========
SEARCH_COLLECTION = "search_results_collection"


def _summarize_search_results_to_json(query: str, results: list[dict]) -> dict | None:
    """调用 LLM 将搜索结果总结为结构化 JSON。

    返回格式:
        {"query": str, "summary": str, "key_points": [str, ...], "sources": [{"title": str, "url": str}, ...]}
    失败时返回 None。
    """
    raw_text = "\n".join(
        f"- {r.get('title', '')}：{r.get('content', '')}" for r in results
    )
    prompt = (
        "你是一个信息提取专家。请将以下搜索结果总结为一个 JSON 对象，格式如下：\n"
        '{"query": "原始搜索词", "summary": "一段 100 字以内的综合摘要", '
        '"key_points": ["要点1", "要点2", ...], '
        '"sources": [{"title": "标题", "url": "链接"}, ...]}\n'
        "只返回 JSON，不要包含任何其他文字。\n\n"
        f"搜索词：{query}\n搜索结果：\n{raw_text}"
    )
    try:
        resp = client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        content = resp.choices[0].message.content.strip()
        # 兼容 LLM 返回 ```json ... ``` 包裹的情况
        if content.startswith("```"):
            content = content.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
        data = json.loads(content)
        logger.info(f"[Web Search] LLM 总结成功，key_points 数量: {len(data.get('key_points', []))}")
        return data
    except Exception as e:
        logger.error(f"[Web Search] LLM 总结失败: {e}")
        return None


def _store_search_summary_to_milvus(summary_json: dict) -> None:
    """将 LLM 总结后的搜索结果增量写入 Milvus 向量数据库。"""
    try:
        text = json.dumps(summary_json, ensure_ascii=False)
        emb = embedding_model.encode_documents([text])
        embedding_dim = len(emb[0])

        milvus_client = MilvusClient(uri=MILVUS_URI)

        # 检测已有 Collection 的 metric_type，不匹配则删除重建
        if milvus_client.has_collection(SEARCH_COLLECTION):
            indexes = milvus_client.list_indexes(collection_name=SEARCH_COLLECTION)
            if indexes:
                idx_info = milvus_client.describe_index(collection_name=SEARCH_COLLECTION, index_name=indexes[0])
                if idx_info.get('metric_type', '') not in ('', 'COSINE'):
                    milvus_client.drop_collection(SEARCH_COLLECTION)
                    logger.warning('[Web Search] 已删除旧 search Collection (metric_type 不匹配)，将重新创建')

        if not milvus_client.has_collection(SEARCH_COLLECTION):
            milvus_client.create_collection(
                collection_name=SEARCH_COLLECTION,
                dimension=embedding_dim,
                metric_type="COSINE",
                consistency_level="Strong",
            )

        # 去重：相似度 >= 0.6 视为已存在
        search_res = milvus_client.search(
            collection_name=SEARCH_COLLECTION,
            data=emb,
            limit=1,
            search_params={"metric_type": "COSINE", "params": {}},
            output_fields=["text"],
        )
        if search_res and search_res[0]:
            top_score = search_res[0][0]["distance"]
            if top_score >= UPSERT_SIMILARITY_THRESHOLD:
                matched_text = search_res[0][0].get("entity", {}).get("text", "未知")
                logger.info(
                    f"[Web Search] 搜索结果已存在 (相似度 {top_score:.3f})，跳过写入。"
                    f"匹配到的已有记录: {matched_text[:200]}"
                )
                return

        # 分配新 id
        existing = milvus_client.query(
            collection_name=SEARCH_COLLECTION, filter="", output_fields=["id"], limit=10000
        )
        next_id = max((r["id"] for r in existing), default=-1) + 1

        milvus_client.insert(
            collection_name=SEARCH_COLLECTION,
            data=[{"id": next_id, "vector": emb[0], "text": text}],
        )
        milvus_client.flush(collection_name=SEARCH_COLLECTION)
        logger.info(f"[Web Search] 搜索结果已写入 Milvus (id={next_id})")
    except Exception as e:
        logger.error(f"[Web Search] 写入 Milvus 失败: {e}")


def real_search_web(query: str) -> str:
    """调用 Tavily API 搜索网页，LLM 总结后存入向量数据库，返回前 5 条结果的摘要。"""
    try:
        logger.info(f"[Web Search] 搜索关键词: {query}")
        tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        response = tavily_client.search(query=query, max_results=5)
        results = response.get("results", [])
        if not results:
            logger.warning(f"[Web Search] 未找到结果: {query}")
            return f"未找到关于 '{query}' 的搜索结果。"

        # --- LLM 总结 + 存入向量数据库 ---
        summary_json = _summarize_search_results_to_json(query, results)
        if summary_json:
            _store_search_summary_to_milvus(summary_json)

        # --- 原有格式化返回给 Agent ---
        formatted = []
        for i, r in enumerate(results, 1):
            title = r.get("title", "无标题")
            content = r.get("content", "无摘要")
            formatted.append(f"{i}. {title}\n   {content}")
        logger.info(f"[Web Search] 返回 {len(results)} 条结果")
        return "\n".join(formatted)
    except Exception as e:
        logger.error(f"[Web Search] 搜索失败: {e}")
        return f"网页搜索失败: {str(e)}"

In [ ]:
def real_query_product_database(product_name: str) -> str:
    """通过 Milvus 向量数据库语义检索商品信息。"""
    try:
        logger.info(f"[Product DB] 查询商品: {product_name}")
        milvus_client = MilvusClient(uri=MILVUS_URI)

        query_embedding = embedding_model.encode_queries([product_name])

        search_res = milvus_client.search(
            collection_name=PRODUCT_COLLECTION,
            data=query_embedding,
            limit=1,
            search_params={"metric_type": "COSINE", "params": {}},
            output_fields=["text"],
        )

        if not search_res or not search_res[0]:
            logger.warning(f"[Product DB] 未找到匹配: {product_name}")
            return f"产品数据库中未找到与 '{product_name}' 匹配的商品。"

        top_result = search_res[0][0]
        if top_result["distance"] < SIMILARITY_THRESHOLD:
            logger.warning(f"[Product DB] 相似度过低 ({top_result['distance']:.3f}): {product_name}")
            return f"产品数据库中未找到与 '{product_name}' 匹配的商品。"

        logger.info(f"[Product DB] 匹配成功，相似度: {top_result['distance']:.3f}")
        return top_result["entity"]["text"]
    except Exception as e:
        logger.error(f"[Product DB] 查询失败: {e}")
        return f"商品数据查询失败: {str(e)}"

In [ ]:
import json

def real_generate_emoji(context: str) -> list:
    """调用 DeepSeek LLM 生成与上下文匹配的表情符号。"""
    try:
        logger.info(f"[Emoji Gen] 生成表情，上下文: {context[:50]}...")
        response = client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "你是一个表情符号专家。根据用户提供的文案上下文，"
                        "返回 4 到 6 个最匹配的 emoji 表情符号。"
                        "只返回一个 JSON 数组，不要包含任何其他文字。"
                        '示例: ["✨", "💖", "🌊", "💧"]'
                    ),
                },
                {"role": "user", "content": context},
            ],
            temperature=0.7,
        )
        content = response.choices[0].message.content.strip()
        emojis = json.loads(content)
        if isinstance(emojis, list) and 4 <= len(emojis) <= 6:
            logger.info(f"[Emoji Gen] 生成成功: {emojis}")
            return emojis
        logger.warning("[Emoji Gen] LLM 返回格式不符，使用默认表情")
        return DEFAULT_EMOJIS
    except Exception as e:
        logger.error(f"[Emoji Gen] 生成失败: {e}")
        return DEFAULT_EMOJIS

## 5. 工具调度与 Agent 定义

In [ ]:
SYSTEM_PROMPT = """
你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，创作引人入胜、高互动、高转化的笔记文案。

你的任务是根据用户提供的产品和需求，生成包含标题、正文、相关标签和表情符号的完整小红书笔记。

请始终采用'Thought-Action-Observation'模式进行推理和行动。文案风格需活泼、真诚、富有感染力。当完成任务后，请以JSON格式直接输出最终文案，格式如下：
```json
{
  "title": "小红书标题",
  "body": "小红书正文",
  "hashtags": ["#标签1", "#标签2", "#标签3", "#标签4", "#标签5"],
  "emojis": ["✨", "🔥", "💖"]
}
```
在生成文案前，请务必先思考并收集足够的信息。
"""

TOOLS_DEFINITION = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网上的实时信息，用于获取最新新闻、流行趋势、用户评价、行业报告等。请确保搜索关键词精确，避免宽泛的查询。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "要搜索的关键词或问题，例如'最新小红书美妆趋势'或'深海蓝藻保湿面膜 用户评价'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_product_database",
            "description": "查询内部产品数据库，获取指定产品的详细卖点、成分、适用人群、使用方法等信息。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "要查询的产品名称，例如'深海蓝藻保湿面膜'"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_emoji",
            "description": "根据提供的文本内容，生成一组适合小红书风格的表情符号。",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "文案的关键内容或情感，例如'惊喜效果'、'补水保湿'"
                    }
                },
                "required": ["context"]
            }
        }
    }
]

# 将真实工具函数映射到字典
available_tools = {
    "search_web": real_search_web,
    "query_product_database": real_query_product_database,
    "generate_emoji": real_generate_emoji,
}

## 6. Agent 核心引擎

In [ ]:
import re

def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_iterations: int = 10) -> str:
    """
    使用 DeepSeek Agent 生成小红书爆款文案。
    
    Args:
        product_name: 要生成文案的产品名称。
        tone_style: 文案的语气和风格。
        max_iterations: Agent 最大迭代次数，防止无限循环。
        
    Returns:
        生成的爆款文案（JSON 格式字符串）。
    """
    logger.info(f"🚀 启动小红书文案生成助手，产品：{product_name}，风格：{tone_style}")
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为产品「{product_name}」生成一篇小红书爆款文案。要求：语气{tone_style}，包含标题、正文、至少5个相关标签和5个表情符号。请以完整的JSON格式输出，并确保JSON内容用markdown代码块包裹（例如：```json{{...}}```）。"}
    ]
    
    iteration_count = 0
    
    while iteration_count < max_iterations:
        iteration_count += 1
        logger.info(f"-- Iteration {iteration_count} --")
        
        try:
            response = client.chat.completions.create(
                model=DEEPSEEK_MODEL,
                messages=messages,
                tools=TOOLS_DEFINITION,
                tool_choice="auto"
            )

            response_message = response.choices[0].message
            
            if response_message.tool_calls:
                logger.info("Agent: 决定调用工具...")
                messages.append(response_message)
                
                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

                    logger.info(f"Agent Action: 调用工具 '{function_name}'，参数：{function_args}")
                    
                    if function_name in available_tools:
                        tool_function = available_tools[function_name]
                        try:
                            tool_result = tool_function(**function_args)
                        except Exception as e:
                            logger.error(f"[Dispatcher] 工具 '{function_name}' 调用异常: {e}")
                            tool_result = f"工具 '{function_name}' 调用异常: {str(e)}"
                        logger.info(f"Observation: 工具返回结果：{str(tool_result)[:200]}")
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": str(tool_result)
                        })
                    else:
                        error_message = f"错误：未知的工具 '{function_name}'"
                        logger.error(error_message)
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": error_message
                        })
                messages.extend(tool_outputs)
                
            elif response_message.content:
                logger.info(f"[模型生成结果] {response_message.content[:200]}...")
                
                json_string_match = re.search(r"```json\s*(\{.*\})\s*```", response_message.content, re.DOTALL)
                
                if json_string_match:
                    extracted_json_content = json_string_match.group(1)
                    try:
                        final_response = json.loads(extracted_json_content)
                        logger.info("Agent: 任务完成，成功解析最终JSON文案。")
                        return json.dumps(final_response, ensure_ascii=False, indent=2)
                    except json.JSONDecodeError as e:
                        logger.warning(f"Agent: 提取到JSON块但解析失败: {e}")
                        messages.append(response_message)
                else:
                    try:
                        final_response = json.loads(response_message.content)
                        logger.info("Agent: 任务完成，直接解析最终JSON文案。")
                        return json.dumps(final_response, ensure_ascii=False, indent=2)
                    except json.JSONDecodeError:
                        logger.warning("Agent: 生成了非JSON格式内容，继续对话。")
                        messages.append(response_message)
            else:
                logger.warning("Agent: 未知响应，可能需要更多交互。")
                break
                
        except Exception as e:
            logger.error(f"调用 DeepSeek API 时发生错误: {e}")
            break
    
    logger.warning("⚠️ Agent 达到最大迭代次数或未能生成最终文案。")
    return "未能成功生成文案。"

## 7. 输出格式化

In [ ]:
def format_rednote_for_markdown(json_string: str) -> str:
    """
    将 JSON 格式的小红书文案转换为 Markdown 格式，以便于阅读和发布。

    Args:
        json_string: 包含小红书文案的 JSON 字符串。

    Returns:
        格式化后的 Markdown 文本。
    """
    try:
        data = json.loads(json_string)
    except json.JSONDecodeError as e:
        return f"错误：无法解析 JSON 字符串 - {e}\n原始字符串：\n{json_string}"

    title = data.get("title", "无标题")
    body = data.get("body", "")
    hashtags = data.get("hashtags", [])

    markdown_output = f"## {title}\n\n"
    markdown_output += f"{body}\n\n"
    
    if hashtags:
        hashtag_string = " ".join(hashtags)
        markdown_output += f"{hashtag_string}\n"
        
    return markdown_output.strip()

## 8. 端到端测试

In [ ]:
# # 测试案例: 深海蓝藻保湿面膜
# result = generate_rednote("深海蓝藻保湿面膜", "活泼甜美")

# print("--- 格式化后的小红书文案 (Markdown) ---")
# print(format_rednote_for_markdown(result))

In [19]:
# result = generate_rednote("李宁藏蓝色休闲外套", "活泼甜美")

# print("--- 格式化后的小红书文案 (Markdown) ---")
# print(format_rednote_for_markdown(result))

# result = generate_rednote("BACKNOW · 帽子女款2026新款夏季棒球帽显脸小的鸭舌帽原创美式复古百搭软顶", "活泼甜美")
# result = generate_rednote("拉夏贝尔白色蕾丝拼接不规则吊带裙女荷叶边飘带无袖露背外穿裙子", "活泼甜美")
# result = generate_rednote("法式气质吊带背心裙女2026夏新款V领蕾丝飘带温柔风气质百搭短裙", "活泼甜美")
# result = generate_rednote("安嘉枪球摄像头", "严肃科普")
result = generate_rednote("360宠物喂食器", "严肃科普")

print("--- 格式化后的小红书文案 (Markdown) ---")
print(format_rednote_for_markdown(result))

--- 格式化后的小红书文案 (Markdown) ---
## 科学养宠必修课｜360宠物喂食器深度测评，告别喂食焦虑

作为一名宠物营养学研究者，今天我要严肃科普一下智能宠物喂食器的科学价值。

🔬【科学依据】
研究表明，定时定量的喂食方式对宠物健康至关重要。不规律的饮食习惯可能导致宠物肥胖、消化系统疾病等问题。360宠物喂食器通过精准的定时功能，帮助建立科学的喂食节律。

📊【数据支撑】
根据2023年宠物健康报告显示，使用智能喂食器的宠物肥胖率下降37%，消化系统疾病发生率降低42%。这不仅仅是便利，更是科学的健康管理。

📱【远程监控】
360°高清摄像头配合AI识别技术，可以实时监控宠物进食情况。通过手机App，你可以：
1. 查看宠物每次进食量
2. 监测进食速度
3. 记录饮食偏好
4. 及时发现异常进食行为

⏰【精准定时】
支持24小时多时段定时喂食，误差控制在±1分钟内。这对于需要特殊饮食管理的宠物（如糖尿病、肥胖症）尤为重要。

🍽️【卫生保障】
食品级材质+密封设计，有效防止粮食受潮变质。定期自动清洁功能，减少细菌滋生风险。

⚠️【注意事项】
1. 首次使用需逐步过渡，避免宠物不适应
2. 定期检查出粮口是否堵塞
3. 保持设备清洁，建议每周深度清洁一次
4. 备用电源建议配置，防止断电影响喂食

科学养宠不是口号，而是需要落实到每一个细节。360宠物喂食器用科技手段，帮助宠物主人实现真正的科学喂养。

#科学养宠 #智能宠物喂食器 #宠物健康管理 #360宠物喂食器 #养宠必备


## 9. 导出文案为 Markdown 文件

In [ ]:
from datetime import datetime
from pathlib import Path


def export_rednote_to_md(json_string: str, output_dir: str = "md") -> str:
    """将格式化后的小红书文案导出为 Markdown 文件。

    文件名格式: {标题}_{时间戳}.md

    Args:
        json_string: generate_rednote 返回的 JSON 字符串。
        output_dir: 输出目录，相对于当前 notebook 所在路径，默认 'md'。

    Returns:
        生成的文件路径字符串。
    """
    md_content = format_rednote_for_markdown(json_string)

    # 从 JSON 中提取标题用于文件名
    try:
        data = json.loads(json_string)
        title = data.get("title", "未命名文案")
    except Exception:
        title = "未命名文案"

    # 清理文件名中的非法字符
    safe_title = re.sub(r'[\\/:*?\"<>|]', '_', title)[:50]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{safe_title}_{timestamp}.md"

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    file_path = out_path / filename

    file_path.write_text(md_content, encoding="utf-8")
    logger.info(f"[Export] 文案已导出: {file_path}")
    print(f"✅ 文案已导出到: {file_path}")
    return str(file_path)

In [20]:
# 将上面生成的文案导出为 md 文件
export_rednote_to_md(result)

✅ 文案已导出到: md\科学养宠必修课｜360宠物喂食器深度测评，告别喂食焦虑_20260407_213802.md


'md\\科学养宠必修课｜360宠物喂食器深度测评，告别喂食焦虑_20260407_213802.md'